# Campo Eléctrico

El **Campo Eléctrico** ($\vec{E}$) se define como la fuerza electrostática por unidad de carga. Es una magnitud vectorial que describe la perturbación que una carga eléctrica provoca en el espacio que la rodea.

Se define mediante la siguiente expresión:

$$\vec{E} = \frac{\vec{F}}{q}$$

> **Nota sobre la dirección:** Se considera que la carga de prueba es positiva ($q > 0$). Por lo tanto, el vector de fuerza ($\vec{F}$) siempre apuntará en la misma dirección que el vector de campo eléctrico ($\vec{E}$).

---

## Formulación para una Carga Puntual

Para determinar el campo en un punto $P$ ubicado en la posición $\vec{r}$, producido por una carga $q'$ situada en $\vec{r}'$, utilizamos la diferencia vectorial:

$$\vec{E} = \frac{1}{4\pi\epsilon_{0}} \frac{q'}{|\vec{r} - \vec{r}'|^{2}} \frac{(\vec{r} - \vec{r}')}{|\vec{r} - \vec{r}'|}$$


```{image} images_tikz/01_06.svg
:alt: Atracción
:width: 60%
:align: center


Simplificando la expresión, obtenemos la forma estándar para el campo eléctrico en el punto $P$:

$$\vec{E} = \frac{1}{4\pi\epsilon_{0}} q' \frac{(\vec{r} - \vec{r}')}{|\vec{r} - \vec{r}'|^{3}}$$


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, RadioButtons

def interactuar_campo(q_val, x_c, y_c):
    # 1. Crear la rejilla
    x = np.linspace(-5, 5, 20)
    y = np.linspace(-5, 5, 20)
    X, Y = np.meshgrid(x, y)
    
    # 2. Parámetros
    k = 8.99e9
    
    # 3. Cálculo del campo
    dx = X - x_c
    dy = Y - y_c
    r = np.sqrt(dx**2 + dy**2)
    
    # Evitar divisiones por cero cerca de la carga
    r_safe = np.where(r < 0.2, 0.2, r)
    
    Ex = k * q_val * dx / r_safe**3
    Ey = k * q_val * dy / r_safe**3
    
    # Intensidad para el color
    E_mag = np.sqrt(Ex**2 + Ey**2)
    
    # 4. Graficar
    plt.figure(figsize=(8, 7))
    
    # Dibujamos las flechas (normalizadas para que no se amontonen, pero con color por intensidad)
    qvr = plt.quiver(X, Y, Ex/E_mag, Ey/E_mag, np.log10(E_mag), 
                     cmap='plasma', pivot='middle', alpha=0.8)
    
    # Dibujar la carga fuente
    color_c = 'red' if q_val > 0 else 'blue'
    plt.scatter([x_c], [y_c], c=color_c, s=300, edgecolors='black', linewidths=2, zorder=5)
    plt.text(x_c + 0.3, y_c + 0.3, f'q = {q_val*1e6:.1f} µC', fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    plt.title(f'Campo Eléctrico Interactivo: Carga en ({x_c}, {y_c})')
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.grid(True, alpha=0.2)
    plt.colorbar(qvr, label='Intensidad del campo (Escala log$_{10}$)')
    plt.show()

# Configuración de los controles interactivos
interact(interactuar_campo, 
         q_val = RadioButtons(options=[1e-6, -1e-6], value=1e-6, description='Carga:'),
         x_c = FloatSlider(min=-4, max=4, step=0.1, value=0, description='Posición X:'),
         y_c = FloatSlider(min=-4, max=4, step=0.1, value=0, description='Posición Y:'))

interactive(children=(RadioButtons(description='Carga:', options=(1e-06, -1e-06), value=1e-06), FloatSlider(va…

<function __main__.interactuar_campo(q_val, x_c, y_c)>


## Distribuciones Continuas de Carga
En casos donde la carga no es puntual sino que se distribuye en un volumen, superficie o línea, el campo eléctrico se calcula mediante una integración sobre toda la distribución:

$$\vec{E} = \frac{1}{4\pi\epsilon_{0}} \int dq' \frac{(\vec{r} - \vec{r}')}{|\vec{r} - \vec{r}'|^{3}}$$

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def visualizar_dipolo(distancia_separacion):
    # 1. Crear la rejilla de puntos
    x = np.linspace(-5, 5, 25)
    y = np.linspace(-5, 5, 25)
    X, Y = np.meshgrid(x, y)
    
    # 2. Configuración del Dipolo (Cargas fijas en magnitud, móviles en posición)
    q = 1e-6
    k = 8.99e9
    
    # Posiciones de las cargas (centradas respecto al origen)
    pos_p = np.array([distancia_separacion/2, 0])  # Carga positiva
    pos_n = np.array([-distancia_separacion/2, 0]) # Carga negativa
    
    # 3. Función interna para calcular el campo de una carga
    def calcular_E(q_val, r_fuente, X, Y):
        dx = X - r_fuente[0]
        dy = Y - r_fuente[1]
        r = np.sqrt(dx**2 + dy**2)
        r_safe = np.where(r < 0.2, 0.2, r) # Evitar división por cero
        Ex = k * q_val * dx / r_safe**3
        Ey = k * q_val * dy / r_safe**3
        return Ex, Ey

    # 4. Superposición: E_total = E_positivo + E_negativo
    Ex_p, Ey_p = calcular_E(q, pos_p, X, Y)
    Ex_n, Ey_n = calcular_E(-q, pos_n, X, Y)
    
    Ex_total = Ex_p + Ex_n
    Ey_total = Ey_p + Ey_n
    
    # Magnitud para normalizar y dar color
    E_mag = np.sqrt(Ex_total**2 + Ey_total**2)

    # 5. Graficación
    plt.figure(figsize=(9, 8))
    
    # Mapa de vectores (Flechas)
    qvr = plt.quiver(X, Y, Ex_total/E_mag, Ey_total/E_mag, np.log10(E_mag), 
                     cmap='plasma', pivot='middle', alpha=0.8)
    
    # Dibujar las dos cargas
    plt.scatter([pos_p[0]], [pos_p[1]], c='red', s=200, label='$q_+$', edgecolors='black', zorder=5)
    plt.scatter([pos_n[0]], [pos_n[1]], c='blue', s=200, label='$q_-$', edgecolors='black', zorder=5)
    
    plt.title(f'Dipolo Eléctrico: Separación $d$ = {distancia_separacion:.2f} m')
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.legend()
    plt.colorbar(qvr, label='Intensidad del campo (Log$_{10}$)')
    plt.grid(True, alpha=0.2)
    plt.show()

# Slider para separar o juntar las cargas del dipolo
interact(visualizar_dipolo, 
         distancia_separacion = FloatSlider(min=0.5, max=6.0, step=0.1, value=2.0, description='Distancia $d$:'))

interactive(children=(FloatSlider(value=2.0, description='Distancia $d$:', max=6.0, min=0.5), Output()), _dom_…

<function __main__.visualizar_dipolo(distancia_separacion)>

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def visualizar_flujo_dipolo(d_sep):
    # 1. Crear rejilla más fina para que las líneas se vean suaves
    x = np.linspace(-5, 5, 50)
    y = np.linspace(-5, 5, 50)
    X, Y = np.meshgrid(x, y)
    
    k = 8.99e9
    q = 1e-6
    
    # Posiciones del dipolo
    p_pos = np.array([d_sep/2, 0])
    n_pos = np.array([-d_sep/2, 0])
    
    def get_E(q_val, r_f, X, Y):
        dx, dy = X - r_f[0], Y - r_f[1]
        r3 = (np.sqrt(dx**2 + dy**2))**3
        r3 = np.where(r3 < 0.1, 0.1, r3) # Estabilizador
        return k * q_val * dx / r3, k * q_val * dy / r3

    # Superposición
    Ex1, Ey1 = get_E(q, p_pos, X, Y)
    Ex2, Ey2 = get_E(-q, n_pos, X, Y)
    
    Ex, Ey = Ex1 + Ex2, Ey1 + Ey2
    color_v = np.sqrt(Ex**2 + Ey**2)

    # 2. Graficación con Streamlines
    plt.figure(figsize=(10, 8))
    
    # Streamplot crea líneas continuas siguiendo los vectores
    strm = plt.streamplot(X, Y, Ex, Ey, color=np.log10(color_v), 
                          linewidth=1.5, cmap='winter', density=1.5, arrowsize=1.5)
    
    # Dibujar cargas
    plt.scatter([p_pos[0]], [p_pos[1]], c='red', s=250, edgecolors='black', label='$q_+$', zorder=5)
    plt.scatter([n_pos[0]], [n_pos[1]], c='blue', s=250, edgecolors='black', label='$q_-$', zorder=5)
    
    plt.colorbar(strm.lines, label='Intensidad del Campo $\log_{10}(|E|)$')
    plt.title(f'Líneas de Campo Eléctrico: Dipolo ($d={d_sep:.1f}m$)')
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()

interact(visualizar_flujo_dipolo, d_sep=FloatSlider(min=0.5, max=6.0, step=0.1, value=2.0, description='Separación:'))

interactive(children=(FloatSlider(value=2.0, description='Separación:', max=6.0, min=0.5), Output()), _dom_cla…

<function __main__.visualizar_flujo_dipolo(d_sep)>

# Densidad de carga

Para distribuciones continuas de carga, definimos la densidad según la dimensión del objeto:

* **Lineal ($\lambda$):** Se utiliza en objetos unidimensionales (como hilos o varillas).
    $$\lambda = \frac{dq'}{dl} \quad \Rightarrow \quad dq' = \lambda dl$$

* **Superficial ($\sigma$):** Se utiliza en objetos bidimensionales (como placas o láminas).
    $$\sigma = \frac{dq'}{ds} \quad \Rightarrow \quad dq' = \sigma ds$$

* **Volumétrica ($\rho$):** Se utiliza en objetos tridimensionales (como esferas o cubos sólidos).
    $$\rho = \frac{dq'}{dV} \quad \Rightarrow \quad dq' = \rho dV$$

### Guía rápida: ¿Cuál densidad utilizar?

En la resolución de problemas de ingeniería, identificar la geometría es el primer paso para plantear la integral de campo eléctrico.

| Dimensión | Tipo de Densidad | Símbolo | Geometrías Comunes | Diferencial de carga ($dq'$) |
| :--- | :--- | :---: | :--- | :--- |
| **1D** | Lineal | $\lambda$ | Hilos, anillos, varillas delgadas. | $dq' = \lambda dl$ |
| **2D** | Superficial | $\sigma$ | Discos, láminas, cáscaras esféricas. | $dq' = \sigma ds$ |
| **3D** | Volumétrica | $\rho$ | Esferas sólidas, cilindros macizos. | $dq' = \rho dV$ |

---

> **💡 Consejo para el cálculo:** > Recuerda que si la carga está distribuida de manera **uniforme**, puedes calcular la densidad como la carga total dividida por la longitud, área o volumen total ($Q/L$, $Q/A$ o $Q/V$).

In [18]:
import ipywidgets as widgets
from IPython.display import display, Math

def asistente_planteamiento(Geometría):
    print(f"--- Análisis para: {Geometría} ---")
    
    if Geometría == "Hilo / Varilla (1D)":
        display(Math(r"\text{Diferencial: } dq' = \lambda \, dl"))
        display(Math(r"\text{Integral: } \vec{E} = \frac{1}{4\pi\epsilon_0} \int \frac{\lambda \, dl}{r^2} \hat{r}"))
        print("\nEjemplo: Si el hilo está en el eje x, dl = dx'.")
        
    elif Geometría == "Anillo (1D)":
        display(Math(r"\text{Diferencial: } dq' = \lambda \, R \, d\phi'"))
        display(Math(r"\text{Integral: } \vec{E} = \frac{1}{4\pi\epsilon_0} \int_{0}^{2\pi} \frac{\lambda R \, d\phi'}{r^2} \hat{r}"))
        print("\nEjemplo: Útil para calcular el campo en el eje de simetría.")

    elif Geometría == "Disco / Lámina (2D)":
        display(Math(r"\text{Diferencial: } dq' = \sigma \, dA' = \sigma (r' \, dr' \, d\theta')"))
        display(Math(r"\text{Integral: } \vec{E} = \frac{1}{4\pi\epsilon_0} \iint \frac{\sigma \, dA'}{r^2} \hat{r}"))
        print("\nEjemplo: En coordenadas polares, dA' es el área de un pequeño sector.")

    elif Geometría == "Esfera Sólida (3D)":
        display(Math(r"\text{Diferencial: } dq' = \rho \, dV' = \rho (r'^2 \sin\theta' \, dr' \, d\theta' \, d\phi')"))
        display(Math(r"\text{Integral: } \vec{E} = \frac{1}{4\pi\epsilon_0} \iiint \frac{\rho \, dV'}{r^2} \hat{r}"))
        print("\nEjemplo: Para distribuciones uniformes, la Ley de Gauss suele ser más rápida.")

# Crear menú interactivo
geometrias = [
    "Hilo / Varilla (1D)", 
    "Anillo (1D)", 
    "Disco / Lámina (2D)", 
    "Esfera Sólida (3D)"
]

widgets.interact(asistente_planteamiento, Geometría=geometrias);

interactive(children=(Dropdown(description='Geometría', options=('Hilo / Varilla (1D)', 'Anillo (1D)', 'Disco …